# Tech-Triathlon 2026: Datathon Master Notebook
**Team:** Nexora  
**Repository:** `saadmaz/Nexora_Waypoint` | **Branch:** `feature/datathon`  
**Competition Deadline:** Friday 9 Oct 2026, 23:59 Asia/Colombo  

---

### Executive Overview & Deliverables
This notebook implements the complete, end-to-end data science and operational research pipeline for the Tech-Triathlon 2026 Datathon:
1. **Task 1: Outlet Service Duration & Arrival Lateness Probability**
   - Exact label construction stripping early-arrival wait times (`max(arrival, open)`).
   - Hard runtime leakage boundary gate (`assert_no_leakage`) and 5-fold out-of-fold target encodings.
   - Dual-head LightGBM models delivering **4.001 min Service MAE** (+46.3% over published standard) and **0.1537 Lateness LogLoss** (+68.9% over historical prior, **0.9762 ROC-AUC**).
2. **Task 2A: 10-Week Depot $\times$ Brand Demand Forecasting (2026-W14 to W23)**
   - Continuous 117-week historical series integration (2024-W01 to 2026-W13).
   - Calendar decomposition modeling the Sinhala & Tamil New Year festive surge (W15) and post-holiday dip (W16).
   - Domain-constrained multi-series ensemble (strictly 0.0 chilled volume for Style & Tech).
3. **Task 2B: Peak-Day Fleet Allocation & Deferral Policy (Scenario S1)**
   - Exact Mixed-Integer Linear Program (MILP) solved with HiGHS to **0.00% optimality gap**.
   - Achieves **79 served out of 85 orders** (92.9% fulfillment rate) across 23 trips.
   - Validated against official `check_allocation.py` with **0 feasibility violations**.


## 1. Setup, Environment & Path Verification
We verify that external competition datasets are located outside git and that the runtime environment satisfies all dependencies.


In [1]:
import sys
import os
from pathlib import Path

# Add src package to sys.path
repo_root = Path.cwd()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import subprocess

from nexora_dt.paths import DATASET_ROOT, TRAIN, TEST, GENERAL, TEMPLATES, ARTIFACTS, CHECKER, assert_data_external
from nexora_dt.leakage import assert_no_leakage

# Hard guard: Ensure datasets are strictly external to git repository
assert_data_external()
print(f"Dataset root verified: {DATASET_ROOT}")
print(f"Checker script verified: {CHECKER}")
print("Dependencies loaded successfully!")


Dataset root verified: /Users/walapalam/Development/TechTriathlon2026_Datasets/Tech-Triathlon 2026 - Datasets
Checker script verified: /Users/walapalam/Development/TechTriathlon2026_Datasets/Tech-Triathlon 2026 - Datasets/check_allocation.py
Dependencies loaded successfully!


## 2. Task 1: Label Construction & Exploratory Data Analysis
Per official competition rules:
- *"Outlets receive goods only within their delivery window. A vehicle that arrives early waits until the window opens."*
$$\text{effective\_start} = \max(\text{arrival\_time}, \text{window\_open\_time})$$
$$\text{service\_time\_min} = \text{leave\_outlet\_time} - \text{effective\_start}$$
- *"Lateness refers to arrival after the window closes."*
$$\text{late\_binary} = \mathbb{I}(\text{arrival\_time} > \text{window\_close\_time})$$


In [2]:
from nexora_dt.labels_task1 import load_joined, build_labels, hhmm_to_min

# Load training data and compute labels
m_train = load_joined("train")
labels_train = build_labels("train")

arr_min = hhmm_to_min(m_train["arrival_time"])
open_min = hhmm_to_min(m_train["window_open_time"])
close_min = hhmm_to_min(m_train["window_close_time"])

early_arrivals = (arr_min < open_min).sum()
wait_times = (open_min - arr_min)[arr_min < open_min]
late_arrivals = labels_train["late_binary"].sum()
total_serviced = len(labels_train)

print(f"Total evaluated route legs: {total_serviced:,}")
print(f"Early arrivals (waited for window): {early_arrivals:,} ({early_arrivals/total_serviced*100:.2f}%)")
print(f"Mean wait time for early arrivals: {wait_times.mean():.1f} min (Max wait: {wait_times.max()} min)")
print(f"Late arrivals (after window close): {late_arrivals:,} ({late_arrivals/total_serviced*100:.2f}%)")
print(f"Service duration min/median/mean/max: {labels_train['service_time_min'].min()} / {labels_train['service_time_min'].median():.1f} / {labels_train['service_time_min'].mean():.2f} / {labels_train['service_time_min'].max()} min")


Total evaluated route legs: 91,894
Early arrivals (waited for window): 4,023 (4.38%)
Mean wait time for early arrivals: 20.0 min (Max wait: 132 min)
Late arrivals (after window close): 17,991 (19.58%)
Service duration min/median/mean/max: 2 / 15.0 / 19.00 / 428 min


## 3. Task 1: Feature Engineering & Leakage Boundary Verification
All engineered features utilize approved prefixes (`ord_`, `plan_`, `win_`, `route_`, `veh_`, `cal_`, `geo_`, `hist_`).
Target encodings are computed strictly **out-of-fold** on the training partition and evaluated through `assert_no_leakage()`.


In [3]:
from nexora_dt.features_task1 import extract_raw_features, add_target_encodings

print("Extracting raw feature tables...")
tr_raw = extract_raw_features("train")
te_raw = extract_raw_features("test")

print("Generating out-of-fold target encodings & verifying leakage gate...")
X_train, X_test = add_target_encodings(tr_raw, te_raw, labels_train, n_splits=5, seed=42)

feature_cols = [c for c in X_train.columns if c != "delivery_id"]
print(f"Verified feature count: {len(feature_cols)} features.")
print(f"Null values in train: {X_train.isna().sum().sum()}, Null values in test: {X_test.isna().sum().sum()}")
print("Hard leakage verification: PASSED!")


Extracting raw feature tables...


Generating out-of-fold target encodings & verifying leakage gate...
Verified feature count: 54 features.


Null values in train: 0, Null values in test: 0
Hard leakage verification: PASSED!


## 4. Task 1: Model Training, 5-Fold Cross-Validation & Baseline Comparison
We train two LightGBM models:
1. **Service Time Regressor:** Optimizes $L_1$ Mean Absolute Error (`regression_l1`).
2. **Arrival Lateness Classifier:** Optimizes Binary LogLoss (`binary_logloss`).

We benchmark directly against the competition baselines:
- **Service Allowance Baseline:** Published standard handling allowances from `service_allowance.csv`.
- **Lateness Baseline:** Historical late prior rate (19.58%).


In [4]:
from nexora_dt.train_task1 import train_and_predict

# Execute 5-Fold training and prediction pipeline
sub_task1, summary_task1 = train_and_predict(n_splits=5, seed=42)

print("\n--- FINAL BENCHMARK SUMMARY (TASK 1) ---")
print(f"Service Time MAE:     {summary_task1['overall_svc_mae']:.3f} min (Baseline: {summary_task1['base_svc_mae']:.3f} min -> {summary_task1['svc_gain_pct']:+.1f}% improvement)")
print(f"Lateness LogLoss:     {summary_task1['overall_late_loss']:.4f} (Baseline: {summary_task1['base_late_loss']:.4f} -> {summary_task1['late_gain_pct']:+.1f}% improvement)")
print(f"Lateness ROC-AUC:     {summary_task1['overall_late_auc']:.4f}")


STEP 1: Extracting Raw Features & Ground Truth Labels


Computing out-of-fold target encodings and running leakage verification...


Dataset summary: 91,894 train rows, 5,014 test rows, 54 features.

--- BASELINE METRICS ---
Baseline Service MAE (Published Service Allowance): 7.451 min
Baseline Lateness LogLoss (Historical Prior 0.1958): 0.4945
Baseline Lateness Brier Score: 0.1575

STEP 2: Training 5-Fold Cross-Validated Models


/Users/walapalam/Development/Nexora_Waypoint/datathon/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


/Users/walapalam/Development/Nexora_Waypoint/datathon/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

/Users/walapalam/Development/Nexora_Waypoint/datathon/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4/5 | Service MAE: 4.018m (vs 7.460m base) | Late LogLoss: 0.1556 | Late AUC: 0.9762


/Users/walapalam/Development/Nexora_Waypoint/datathon/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


/Users/walapalam/Development/Nexora_Waypoint/datathon/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5/5 | Service MAE: 4.045m (vs 7.430m base) | Late LogLoss: 0.1521 | Late AUC: 0.9769

STEP 3: Overall Out-Of-Fold Evaluation vs Baselines
Service Time MAE:     4.001 min (Baseline: 7.451 min -> +46.3% improvement)
Lateness LogLoss:     0.1537 (Baseline: 0.4945 -> +68.9% improvement)
Lateness ROC-AUC:     0.9762
Lateness Brier Score: 0.0479 (Baseline: 0.1575)

Top 5 Service Time Features:
  - hist_outlet_mean_service: 1099.2
  - cal_iso_week: 932.6
  - ord_units: 852.6
  - hist_outlet_late_rate: 825.6
  - ord_volume_per_unit: 809.4

Top 5 Lateness Probability Features:
  - win_slack_min: 935.6
  - geo_disruption_index: 851.8
  - hist_outlet_late_rate: 551.6
  - cal_iso_week: 509.0
  - route_seq: 498.0

Wrote verified submission to:
  - /Users/walapalam/Development/TechTriathlon2026_Datasets/artifacts/submission_task1.csv
  - /Users/walapalam/Development/Nexora_Waypoint/datathon/submissions/submission_task1.csv

--- FINAL BENCHMARK SUMMARY (TASK 1) ---
Service Time MAE:     4.001 mi

### Task 1 Single-Row Inference Demonstration
Below is an explicit demonstration of evaluating the dual-head inference pipeline on an arbitrary single delivery order.


In [5]:
# Inspect sample prediction from test inputs
sample_idx = 10
sample_row = sub_task1.iloc[sample_idx]
print(f"Sample Delivery ID:          {sample_row['delivery_id']}")
print(f"Predicted Service Duration:  {sample_row['pred_service_min']:.2f} minutes")
print(f"Predicted Late Probability:  {sample_row['pred_late_prob']*100:.2f}%")


Sample Delivery ID:          ORD0092318
Predicted Service Duration:  7.62 minutes
Predicted Late Probability:  0.01%


## 5. Task 2A: 10-Week Depot $\times$ Brand Demand Forecasting
We forecast weekly delivered volume for 10 weeks (2026-W14 through 2026-W23) across 6 time series:
- Depots: `Kandy`, `Peliyagoda`
- Brands: `Fresh`, `Style`, `Tech`

The pipeline integrates 117 continuous weeks of historical data, modeling the Sinhala & Tamil New Year retail surge (W15) and post-holiday dip (W16).
Chilled volume for Style and Tech is strictly constrained to 0.000, while Fresh maintains the empirical depot chilled ratio (~36.5%).


In [6]:
from nexora_dt.train_task2a import run_task2a_pipeline

sub_task2a, summary_task2a = run_task2a_pipeline()
print("\nGenerated Task 2A Forecast (First 12 rows):")
display(sub_task2a.head(12))


TASK 2A: 10-Week Demand Forecasting Pipeline



Forecast summary by (depot, brand):
                  avg_total_vol  max_total_vol  avg_chilled_vol
depot      brand                                               
Kandy      Fresh       540.3422        583.876         196.3800
           Style        80.7079        102.468           0.0000
           Tech         23.3705         27.897           0.0000
Peliyagoda Fresh      1024.8101       1120.331         377.3882
           Style       148.4269        178.176           0.0000
           Tech         34.7369         39.308           0.0000

Wrote Task 2A submission to:
  - /Users/walapalam/Development/TechTriathlon2026_Datasets/artifacts/submission_task2a.csv
  - /Users/walapalam/Development/Nexora_Waypoint/datathon/submissions/submission_task2a.csv

Generated Task 2A Forecast (First 12 rows):


,row_id,pred_total_volume_m3,pred_chilled_volume_m3
0,W0000,547.023,198.808
1,W0001,86.340,0.000
2,W0002,24.322,0.000
3,W0003,1041.703,383.609
4,W0004,140.471,0.000
5,W0005,37.713,0.000
6,W0006,583.876,212.202
7,W0007,102.468,0.000
8,W0008,22.973,0.000
9,W0009,1120.331,412.564


## 6. Task 2B: Peak-Day Fleet Allocation & Deferral Policy
For Scenario S1, 85 orders were requested at Peliyagoda with 28 available fleet vehicles and 10 in workshop maintenance.
We formulate an exact Mixed-Integer Linear Program (MILP) with HiGHS solving all 7 competition feasibility rules and daily budgets:
- Fresh pre-dawn budget $\le 270$ min (03:30 to 08:00)
- Style/Tech daytime budget $\le 480$ min
- Max 2 trips per vehicle per day
- Single brand & single district per trip
- Reefer required for chilled; van required for van-only access


In [7]:
from nexora_dt.alloc2b import solve
from nexora_dt.paths import CHECKER

out_task2b = ARTIFACTS / "submission_task2b.csv"
exit_code = solve("S1", out_task2b)

# Run official check_allocation.py gate
res = subprocess.run([sys.executable, str(CHECKER), str(out_task2b)], capture_output=True, text=True)
print(res.stdout)
assert res.returncode == 0, "check_allocation.py failed!"


Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
MIP has 5753 rows; 4072 cols; 27776 nonzeros; 4072 integer variables (4072 binary)
Coefficient ranges:
  Matrix  [5e-01, 7e+03]
  Cost    [1e-02, 2e+02]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 5e+02]
Presolving model
5595 rows, 3934 cols, 26820 nonzeros 0s
4690 rows, 3708 cols, 24786 nonzeros 0s
Presolve reductions: rows 4690(-1063); columns 3708(-364); nonzeros 24786(-2990) 


Objective function is integral with scale 100

Solving MIP model with:
   4690 rows
   3708 cols (3708 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   24786 nonzeros
   Thread count 5 (of 10 threads). Using 1 max workers. Parallel search off

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constraints |       Work      
Src  Proc. InQueue |  Leaves   Expl. | BestBound       BestSol              Gap |   Cuts   InLp Confl. | LpIters     Time

 z       0       0         0   0.00%   -inf            0                  Large        0      0      0         0  

 L       0       0         0   0.00%   -3054.235561    -2818.77           8.35%     1747    100   3615     10273     3.2s

Symmetry detection completed in 0.1s
Found 39 generator(s) and 6 full orbitope(s) acting on 892 columns



 L       0       0         0   0.00%   -3054.235561    -2821.77           8.24%     1041     58   3615     11923     4.1s


         5       0         2  25.00%   -3054.235561    -2821.77           8.24%     1047     58   3708     67586     9.4s


 L      20       1         6  46.88%   -3049.65547     -2922.77           4.34%     1816    109   3940    124774    16.8s


        57       4        20  83.06%   -3021.274371    -2922.77           3.37%     2330    139   4275    187744    22.4s


        91       0        41 100.00%   -2922.798341    -2922.77           0.00%     2354    139   4609    219565    26.2s

Solving report
  Status            Optimal
  Primal bound      -2922.77
  Dual bound        -2922.79
  Gap               0.000684% (tolerance: 0.01%)
  P-D integral      5.07589260607
  Solution status   feasible
                    -2922.77 (objective)
                    0 (bound viol.)
                    0 (int. viol.)
                    0 (row viol.)
  Timing            26.17
                    0.29 (Presolve)
                        MIP    time [calls] = 0.10 [1]
                        subMIP time [calls] = 0.20 [28]
                    25.88 (Solve)
                        MIP    time [calls] = 23.13 [1]
                        subMIP time [calls] = 2.75 [28]
                    0.00 (Postsolve)
                        MIP    time [calls] = 0.00 [1]
                        subMIP time [calls] = 0.00 [28]
  Max sub-MIP depth 16
  Nodes             91
  Rep

FEASIBILITY: PASSED - every rule satisfied.


FEASIBILITY: PASSED - every rule satisfied.



### Task 2B Deferral Analysis & Categorization
Under global optimality, **79 orders are served and exactly 6 orders are deferred**.
We analyze the physical necessity of each deferral below:


In [8]:
scn = pd.read_csv(TEST / "task2b_peak_day_scenarios.csv")
sub_2b = pd.read_csv(out_task2b)
merged_2b = scn.merge(sub_2b, on=["scenario", "order_ref"])
deferred = merged_2b[merged_2b["decision"] == "deferred"][
    ["order_ref", "brand", "district", "temp_requirement", "order_weight_kg", "order_volume_m3", "deferred_yesterday"]
]
print(f"Total Deferred Orders: {len(deferred)}")
display(deferred)


Total Deferred Orders: 6


,order_ref,brand,district,temp_requirement,order_weight_kg,order_volume_m3,deferred_yesterday
56,S1-056,Fresh,Galle,chilled,670.7,3.754,0
58,S1-058,Fresh,Galle,chilled,2741.8,16.515,0
64,S1-064,Fresh,Matara,chilled,1277.8,6.784,0
67,S1-067,Fresh,Matara,chilled,930.2,5.194,0
78,S1-078,Style,Kurunegala,ambient,2561.6,40.660,0
83,S1-083,Fresh,Puttalam,chilled,1588.8,8.659,1


## 7. Submission Artifacts & Verification Checklist
We verify that all three competition submission CSV files exist, are strictly formatted, and pass validation gates.


In [9]:
print("VERIFICATION CHECKLIST:")
for name, expected_len in [("submission_task1.csv", 5014), ("submission_task2a.csv", 60), ("submission_task2b.csv", 85)]:
    p = ARTIFACTS / name
    assert p.is_file(), f"Missing {p}"
    df = pd.read_csv(p)
    assert len(df) == expected_len, f"{name}: expected {expected_len} rows, got {len(df)}"
    if "2b" in name:
        assert df["decision"].isna().sum() == 0, f"{name}: contains missing decisions"
        assert df[df.decision == "served"][["vehicle_id", "trip_id"]].isna().sum().sum() == 0, f"{name}: served rows missing vehicle or trip"
        print(f"  [PASSED] {name}: {len(df)} rows, valid decisions, verified on disk.")
    else:
        assert df.isna().sum().sum() == 0, f"{name}: contains NaN values"
        print(f"  [PASSED] {name}: {len(df)} rows, 0 nulls, verified on disk.")

print("\nALL TASKS COMPLETE & VALIDATED FOR TECH-TRIATHLON 2026!")


VERIFICATION CHECKLIST:
  [PASSED] submission_task1.csv: 5014 rows, 0 nulls, verified on disk.
  [PASSED] submission_task2a.csv: 60 rows, 0 nulls, verified on disk.
  [PASSED] submission_task2b.csv: 85 rows, valid decisions, verified on disk.

ALL TASKS COMPLETE & VALIDATED FOR TECH-TRIATHLON 2026!
